# Assemble per-run summary and ph csv files

Reference: `20251113_assemble_summary_ph_csv.ipynb`

In [1]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
path_main = Path("../simulations")

path_data = path_main / "20251118"

path_save = path_main / "20251118_summary"
if not path_save.exists():
    path_save.mkdir(parents=True, exist_ok=True)

In [3]:
def load_summary(search_type, cr, br, bmr, pm):
    dates = ["20251118", "20251119"]
    files = []
    for d in dates:
        folder = f"{d}_{search_type}_pmaxRepeat_canvasRadius{cr}_beamMovementRadius{bmr}_beamRadius{br}_pm{pm:.0e}_pfa{pm:.0e}"
        files += sorted(list((path_data / folder).glob("*_summary.csv")))

    data_list = []
    for f in files:
        data_list.append(pd.read_csv(f, index_col=0))
    df = pd.concat(data_list).reset_index(drop=True)
    return df

In [4]:
def load_ph(search_type, cr, br, bmr, pm):
    dates = ["20251118", "20251119"]
    files = []
    for d in dates:
        folder = f"{d}_{search_type}_pmaxRepeat_canvasRadius{cr}_beamMovementRadius{bmr}_beamRadius{br}_pm{pm:.0e}_pfa{pm:.0e}"
        files += sorted(list((path_data / folder).glob("*_ph.csv")))

    p_list = []
    h_list = []
    for f in files:
        df = pd.read_csv(f, index_col=0)
        p_list.append(df["p_max_all"])
        h_list.append(df["h_actual_all"])

    # If less than 1000 runs, print out missing run
    if len(h_list) < 1000:
        run_nums = []
        for f in files:
            a = re.match(r"infotaxis_(\d{4})_ph.csv", f.name)
            run_nums.append(int(a.groups()[0]))
        missing_runs = set(range(1, 1001)) - set(run_nums)
        print(f"Missing runs for cr={cr}, br={br}, pm={pm:.0e}: {missing_runs}")

    # Pad ragged list to array
    len_max = max(len(h) for h in h_list)
    h_actual_pad = np.array([h.tolist() + [np.nan] * (len_max - len(h)) for h in h_list])
    p_max_pad = np.array([p.tolist() + [np.nan] * (len_max - len(p)) for p in p_list])

    ds = xr.Dataset(
        data_vars=dict(
            h_actual=(["run", "ping"], h_actual_pad),
            p_max=(["run", "ping"], p_max_pad),
        ),
        coords=dict(
            run=("run", range(1, len(h_list) + 1)),
            ping=("ping", range(len_max)),
        ),
    )
    return ds 

## PM=PFA sweep

In [5]:
cr_all = [5, 10]
br_all = [1, 2]
bmr_all = [2, 3]
pm_all = [0.001, 0.01, 0.02, 0.05]

In [6]:
# infotaxis ph data
search_type = "infotaxis"
for cr in cr_all:
    for br in br_all:
        for bmr in bmr_all:
            for pm in pm_all:
                print(f"cr={cr}, br={br}, bmr={bmr}, pm={pm}")
                ds = load_ph(search_type, cr=cr, br=br, bmr=bmr, pm=pm)
                ds.to_netcdf(path_save / f"{search_type}_cr{cr}_br{br}_bmr{bmr}_pm{pm:.0e}_pfa{pm:.0e}_ph.nc")

cr=5, br=1, bmr=2, pm=0.001
cr=5, br=1, bmr=2, pm=0.01
cr=5, br=1, bmr=2, pm=0.02
cr=5, br=1, bmr=2, pm=0.05
cr=5, br=1, bmr=3, pm=0.001
cr=5, br=1, bmr=3, pm=0.01
cr=5, br=1, bmr=3, pm=0.02
cr=5, br=1, bmr=3, pm=0.05
cr=5, br=2, bmr=2, pm=0.001
cr=5, br=2, bmr=2, pm=0.01
cr=5, br=2, bmr=2, pm=0.02
cr=5, br=2, bmr=2, pm=0.05
cr=5, br=2, bmr=3, pm=0.001
cr=5, br=2, bmr=3, pm=0.01
cr=5, br=2, bmr=3, pm=0.02
cr=5, br=2, bmr=3, pm=0.05
cr=10, br=1, bmr=2, pm=0.001
cr=10, br=1, bmr=2, pm=0.01
cr=10, br=1, bmr=2, pm=0.02
cr=10, br=1, bmr=2, pm=0.05
cr=10, br=1, bmr=3, pm=0.001
cr=10, br=1, bmr=3, pm=0.01
cr=10, br=1, bmr=3, pm=0.02
cr=10, br=1, bmr=3, pm=0.05
cr=10, br=2, bmr=2, pm=0.001
cr=10, br=2, bmr=2, pm=0.01
cr=10, br=2, bmr=2, pm=0.02
cr=10, br=2, bmr=2, pm=0.05
cr=10, br=2, bmr=3, pm=0.001
cr=10, br=2, bmr=3, pm=0.01
cr=10, br=2, bmr=3, pm=0.02
cr=10, br=2, bmr=3, pm=0.05


In [7]:
# infotaxis summary
search_type = "infotaxis"
for cr in cr_all:
    for br in br_all:
        for bmr in bmr_all:
            for pm in pm_all:
                print(f"cr={cr}, br={br}, bmr={bmr}, pm={pm}")
                df = load_summary(search_type, cr=cr, br=br, bmr=bmr, pm=pm)
                df.to_csv(path_save / f"{search_type}_cr{cr}_br{br}_bmr{bmr}_pm{pm:.0e}_pfa{pm:.0e}_summary.csv")

cr=5, br=1, bmr=2, pm=0.001
cr=5, br=1, bmr=2, pm=0.01
cr=5, br=1, bmr=2, pm=0.02
cr=5, br=1, bmr=2, pm=0.05
cr=5, br=1, bmr=3, pm=0.001
cr=5, br=1, bmr=3, pm=0.01
cr=5, br=1, bmr=3, pm=0.02
cr=5, br=1, bmr=3, pm=0.05
cr=5, br=2, bmr=2, pm=0.001
cr=5, br=2, bmr=2, pm=0.01
cr=5, br=2, bmr=2, pm=0.02
cr=5, br=2, bmr=2, pm=0.05
cr=5, br=2, bmr=3, pm=0.001
cr=5, br=2, bmr=3, pm=0.01
cr=5, br=2, bmr=3, pm=0.02
cr=5, br=2, bmr=3, pm=0.05
cr=10, br=1, bmr=2, pm=0.001
cr=10, br=1, bmr=2, pm=0.01
cr=10, br=1, bmr=2, pm=0.02
cr=10, br=1, bmr=2, pm=0.05
cr=10, br=1, bmr=3, pm=0.001
cr=10, br=1, bmr=3, pm=0.01
cr=10, br=1, bmr=3, pm=0.02
cr=10, br=1, bmr=3, pm=0.05
cr=10, br=2, bmr=2, pm=0.001
cr=10, br=2, bmr=2, pm=0.01
cr=10, br=2, bmr=2, pm=0.02
cr=10, br=2, bmr=2, pm=0.05
cr=10, br=2, bmr=3, pm=0.001
cr=10, br=2, bmr=3, pm=0.01
cr=10, br=2, bmr=3, pm=0.02
cr=10, br=2, bmr=3, pm=0.05
